In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/02/27 11:51:52 WARN Utils: Your hostname, Rob resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/27 11:51:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 11:51:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/27 11:51:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import LongType, DoubleType, StringType, TimestampType
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [4]:
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")

# Disable vectorized reader (important for mixed parquet physical types)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

green_sample = spark.read.parquet("../data/pq/green/2020/01/*")
schema = green_sample.schema

def patch(schema):
    out = []
    for f in schema.fields:
        # IDs ? LongType (safe for INT32 + INT64)
        if f.name in ["VendorID","RatecodeID","PULocationID","DOLocationID","payment_type","trip_type"]:
            out.append(T.StructField(f.name, T.LongType(), True))
        
        # passenger_count ? DoubleType (safe for INT32 + DOUBLE)
        elif f.name == "passenger_count":
            out.append(T.StructField(f.name, T.DoubleType(), True))
        
        else:
            out.append(f)
    
    return T.StructType(out)

green_schema_safe = patch(schema)

df_green = spark.read.schema(green_schema_safe).parquet("../data/pq/green/*/*")

In [5]:
# yellow can stay normal (or do the same approach if it ever errors)
df_yellow = spark.read.parquet("../data/pq/yellow/*/*")

In [6]:
df_green.printSchema()
df_yellow.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- trip_type: long (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |

In [36]:
df_green = (df_green
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
)

In [8]:
df_yellow = (df_yellow
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
)

In [9]:
casts = {
    "VendorID": "long",
    "RatecodeID": "long",
    "PULocationID": "long",
    "DOLocationID": "long",
    "passenger_count": "long",
    "payment_type": "long",
    "trip_type": "long",
    "trip_distance": "double",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
}

for c, t in casts.items():
    if c in df_green.columns:
        df_green = df_green.withColumn(c, F.col(c).cast(t))
    if c in df_yellow.columns:
        df_yellow = df_yellow.withColumn(c, F.col(c).cast(t))

In [10]:
for c in ["pickup_datetime", "dropoff_datetime"]:
    if c in df_green.columns:
        df_green = df_green.withColumn(c, F.col(c).cast("timestamp"))
    if c in df_yellow.columns:
        df_yellow = df_yellow.withColumn(c, F.col(c).cast("timestamp"))

In [11]:
# 4) Compute common columns AFTER standardizing
common_columns = sorted(list(set(df_green.columns) & set(df_yellow.columns)))
# (optional) print to verify
common_columns

['DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'dropoff_datetime',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'pickup_datetime',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance']

In [12]:
common_colums = []
yellow_columns = set(df_yellow.columns)
for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [13]:
common_colums

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge']

In [14]:
# 5) Select common cols + add service_type
df_green_sel = df_green.select(common_columns).withColumn("service_type", F.lit("green"))
df_yellow_sel = df_yellow.select(common_columns).withColumn("service_type", F.lit("yellow"))

In [15]:
# 6) Union safely by name
df_trips_data = df_green_sel.unionByName(df_yellow_sel)

In [16]:
# 7) Sanity check + temp view
df_trips_data.groupBy("service_type").count().show()
df_trips_data.createOrReplaceTempView("trips_data")

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2495901|
|      yellow|55552571|
+------------+--------+



In [17]:
df_trips_data.createOrReplaceTempView("trips_data")

In [18]:
# 6) Union safely by name
df_trips_data = df_green_sel.unionByName(df_yellow_sel)

In [19]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 5:==================================================>      (25 + 3) / 28]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2495901|
|      yellow|55552571|
+------------+--------+



In [20]:
# 7) Sanity check + temp view
df_trips_data.groupBy("service_type").count().show()
df_trips_data.createOrReplaceTempView("trips_data")

[Stage 8:================================================>        (24 + 4) / 28]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2495901|
|      yellow|55552571|
+------------+--------+



In [21]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [22]:
df_green = spark.read.parquet("../data/pq/green/*/*")
df_green.createOrReplaceTempView("green")

df_green_revenue = spark.sql("""
SELECT
  date_trunc('hour', lpep_pickup_datetime) AS hour,
  PULocationID AS zone,
  SUM(total_amount) AS amount,
  COUNT(1) AS number_records
FROM green
WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1, 2
""")

(df_green_revenue
  .coalesce(4)
  .write.mode("overwrite")
  .parquet("../data/report/revenue/green"))

In [23]:
rev = spark.read.parquet("../data/report/revenue/green")
print("rows:", rev.count())
rev.show(5)

rows: 854963
+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-31 15:00:00|  75|1215.5399999999988|            78|
|2020-01-02 19:00:00|  22|              40.8|             1|
|2020-01-05 21:00:00|  41| 323.4200000000001|            26|
|2020-01-18 02:00:00|  92|              91.1|             2|
|2020-01-13 16:00:00|   3|             95.95|             4|
+-------------------+----+------------------+--------------+
only showing top 5 rows



In [24]:
df_yellow = spark.read.parquet("../data/pq/yellow/*/*")
df_yellow.createOrReplaceTempView("yellow")

df_yellow_revenue = spark.sql("""
SELECT
  date_trunc('hour', tpep_pickup_datetime) AS hour,
  PULocationID AS zone,
  SUM(total_amount) AS amount,
  COUNT(1) AS number_records
FROM yellow
WHERE tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1, 2
""")

(df_yellow_revenue
  .coalesce(4)
  .write.mode("overwrite")
  .parquet("../data/report/revenue/yellow"))

In [25]:
rev_y = spark.read.parquet("../data/report/revenue/yellow")
print("rows:", rev_y.count())
rev_y.show(5)

rows: 1915127
+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-29 04:00:00| 234|342.32000000000005|            23|
|2020-01-06 21:00:00| 170|3988.0599999999986|           238|
|2020-01-17 08:00:00|  50|2093.5999999999985|           118|
|2020-01-06 17:00:00| 142|           5255.43|           348|
|2020-01-21 14:00:00| 132|19149.830000000013|           361|
+-------------------+----+------------------+--------------+
only showing top 5 rows



In [26]:
from pyspark.sql import types

green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("ehail_fee", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("trip_type", types.IntegerType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [27]:
from pathlib import Path
from pyspark.sql import functions as F

base = Path("..")
year = 2020
month = 4

input_path  = str(base / f"data/raw/green/{year}/{month:02d}/*.csv.gz")
output_path = str(base / f"data/pq/green/{year}/{month:02d}/")

df = (spark.read
      .option("header", "true")
      .schema(green_schema)
      .csv(input_path))

df = (df
      .withColumn("PULocationID", F.col("PULocationID").cast("long"))
      .withColumn("DOLocationID", F.col("DOLocationID").cast("long")))

(df.repartition(4)
   .write.mode("overwrite")
   .parquet(output_path))

print("rebuilt:", output_path)

rebuilt: ../data/pq/green/2020/04


In [28]:
df_result = spark.sql("""
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records,
    'green' AS service_type
FROM green
GROUP BY 1,2

UNION ALL

SELECT
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records,
    'yellow' AS service_type
FROM yellow
GROUP BY 1,2
""")

In [61]:
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

In [62]:
df_check = spark.read.parquet("data/report/revenue/")
print("rows:", df_check.count())

rows: 2770480


In [63]:
df_check.printSchema()

root
 |-- hour: timestamp (nullable = true)
 |-- zone: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- number_records: long (nullable = true)
 |-- service_type: string (nullable = true)



In [64]:
df_check.show(10)

+-------------------+----+------------------+--------------+------------+
|               hour|zone|            amount|number_records|service_type|
+-------------------+----+------------------+--------------+------------+
|2020-01-12 18:00:00|  41| 916.7399999999996|            66|       green|
|2020-01-30 14:00:00| 225|             93.86|             4|       green|
|2020-01-22 20:00:00|  75| 583.3799999999999|            42|       green|
|2020-01-28 19:00:00|  66| 533.9200000000001|            24|       green|
|2020-01-29 10:00:00|  26|            118.16|             5|       green|
|2020-01-02 12:00:00| 135|137.23000000000002|             5|       green|
|2020-01-27 00:00:00| 255|            116.87|             6|       green|
|2020-01-03 20:00:00| 208|            137.44|             3|       green|
|2020-01-24 04:00:00|  92|             43.73|             2|       green|
|2020-01-04 23:00:00|  66|167.98999999999998|             9|       green|
+-------------------+----+------------

In [65]:
df_check.groupBy("service_type").count().show()

[Stage 90:================================================>         (5 + 1) / 6]

+------------+-------+
|service_type|  count|
+------------+-------+
|       green| 855012|
|      yellow|1915468|
+------------+-------+



In [38]:
df_green = df_green \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

df_yellow = df_yellow \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

In [39]:
print([c for c in df_green.columns if "pickup" in c or "dropoff" in c])
print([c for c in df_yellow.columns if "pickup" in c or "dropoff" in c])

['pickup_datetime', 'dropoff_datetime']
['pickup_datetime', 'dropoff_datetime']


In [40]:
print(sorted(set(common_columns) - set(df_green.columns)))
print(sorted(set(common_columns) - set(df_yellow.columns)))

[]
[]
